A clear prompt gives the LLM specific instructions about its role, the information it must verify, the decision rules it must follow, and the format of the expected response. An unclear prompt uses vague instructions, leaving the LLM to interpret when a refund should be approved or rejected, which can lead to inconsistent decisions.

Two specific changes I would make are:

Add explicit refund decision rules.
For example, instruct the chatbot to first verify the order ID and order status, then check whether the issue meets the refund eligibility conditions before approving or rejecting the request. This matters because the LLM will have defined criteria to follow instead of making decisions based on vague wording.

Specify a fixed output format.
For example, require every response to contain:

Decision: APPROVE or REJECT

Reason: brief explanation

Next Step: action for the customer

This matters because a structured format makes responses consistent and easier for the application or support team to interpret.

Zero-shot: The LLM classifies complaints without examples. It is simple but may be less consistent.

Few-shot: The LLM is given a few labelled examples before the new complaint. This helps it understand the four categories better.

I would choose few-shot prompting because we have 200 examples but a limited context window, so we can select only the most useful examples.

The number of examples depends on:

Context-window size
Length of examples
Coverage of all four categories
Number of confusing/borderline cases
Accuracy and token cost

CoT prompting helps the LLM solve the routing problem step by step by considering traffic delays, priorities, and distances before selecting the final route.

Example prompt:
“Analyze the five delivery stops step by step. Consider distance, traffic delays, and priority levels. Then choose the best stop sequence and briefly explain the factors considered.”

Limitation: CoT can still produce incorrect reasoning and may increase response time and token usage.

I would choose RAG because this use case depends on a policy handbook that is updated every quarter.

Easy updates: With RAG, the updated handbook can simply be re-indexed. There is no need to retrain the LLM every time a policy changes. Fine-tuning would require additional training to incorporate new information reliably.
More accurate and traceable answers: RAG retrieves the relevant sections of the handbook before generating an answer, making it easier to cite and verify the exact policy. Fine-tuning stores knowledge in model parameters, so it may give outdated answers after policy changes.

When fine-tuning would be better: If the goal were to teach the model a specific behavior or output format, such as consistently classifying dispute cases into predefined categories, fine-tuning could be more suitable.

Chunk size determines how much text is stored in each searchable piece, while chunk overlap repeats some text between consecutive chunks to preserve context.

Adjust chunk size: Increase it slightly so related information, such as a dish name and its price, is more likely to appear in the same chunk.
Adjust chunk overlap: Increase the overlap so sentences or important details split across chunk boundaries are still captured together during retrieval.

Trade-off: Larger chunks and more overlap improve context but increase storage and retrieval costs and may return more irrelevant text, making answers less focused.

FAISS: Fast and lightweight vector-search library, ideal when you need high-speed similarity search and can manage data storage/updates yourself.
Best fit: A large, relatively stable embedding collection where performance is the priority.
ChromaDB: A higher-level vector database that provides easier document/metadata management and persistence along with similarity search.
Best fit: Applications where embeddings frequently change and you need convenient filtering and database-style management.

Would switching to ChromaDB fix the outdated complaints?
No. The problem is that resolved records are still present in the index. Changing the vector store does not automatically know which records should be removed.

Required step: Regardless of FAISS or ChromaDB, the system needs a data-cleanup/deletion process that removes or marks resolved complaints as inactive and keeps the vector index synchronized with the source data.

In [1]:
# Structured Prompt Builder for Food Delivery Chatbot

SYSTEM_PROMPT = """
You are a food delivery customer support agent.
Be professional, helpful, and polite.
Keep every response within 80 words.
"""

USER_TEMPLATE = """
Customer Name: {customer_name}
Order ID: {order_id}
Issue Type: {issue_type}

Please help the customer resolve this issue.
"""


def validate_issue_type(issue_type):
    allowed_values = ["late delivery", "missing item", "wrong item"]

    if issue_type not in allowed_values:
        raise ValueError(
            f"Invalid issue type: '{issue_type}'. "
            f"Allowed values are: {', '.join(allowed_values)}."
        )


def build_prompt(customer_name, order_id, issue_type):
    validate_issue_type(issue_type)

    user_prompt = USER_TEMPLATE.format(
        customer_name=customer_name,
        order_id=order_id,
        issue_type=issue_type
    )

    return f"SYSTEM PROMPT:\n{SYSTEM_PROMPT}\nUSER PROMPT:\n{user_prompt}"


# Test cases
test_cases = [
    ("Rahul", "ORD12345", "late delivery"),
    ("Priya", "ORD67890", "missing item"),
    ("Amit", "ORD11111", "wrong address")  # Invalid test case
]

for customer_name, order_id, issue_type in test_cases:
    try:
        prompt = build_prompt(customer_name, order_id, issue_type)
        print("=" * 60)
        print(prompt)

    except ValueError as error:
        print(f"Sorry, the prompt could not be created: {error}")

SYSTEM PROMPT:

You are a food delivery customer support agent.
Be professional, helpful, and polite.
Keep every response within 80 words.

USER PROMPT:

Customer Name: Rahul
Order ID: ORD12345
Issue Type: late delivery

Please help the customer resolve this issue.

SYSTEM PROMPT:

You are a food delivery customer support agent.
Be professional, helpful, and polite.
Keep every response within 80 words.

USER PROMPT:

Customer Name: Priya
Order ID: ORD67890
Issue Type: missing item

Please help the customer resolve this issue.

Sorry, the prompt could not be created: Invalid issue type: 'wrong address'. Allowed values are: late delivery, missing item, wrong item.


In [2]:
# Few-Shot Complaint Classifier Prompt Builder

# Initial labelled examples
examples = [
    {
        "input": "My order arrived 45 minutes late.",
        "output": "Late Delivery"
    },
    {
        "input": "I ordered a veg burger but received a chicken burger.",
        "output": "Wrong Item"
    },
    {
        "input": "My order was missing the fries I paid for.",
        "output": "Missing Item"
    },
    {
        "input": "The food was cold and tasted stale.",
        "output": "Poor Quality"
    }
]


def build_few_shot_prompt(complaint_text):
    prompt = "Classify the food delivery complaint into one of these categories:\n"
    prompt += "Late Delivery, Wrong Item, Missing Item, Poor Quality\n\n"

    # Add few-shot examples
    for example in examples:
        prompt += f"Input: {example['input']}\n"
        prompt += f"Output: {example['output']}\n\n"

    # Add the new complaint
    prompt += f"Input: {complaint_text}\n"
    prompt += "Output:"

    return prompt


def add_example(text, label):
    examples.append({
        "input": text,
        "output": label
    })


# Test 1
complaint1 = "My delivery arrived one hour after the promised time."
print("=" * 60)
print(build_few_shot_prompt(complaint1))

# Test 2
complaint2 = "The pizza I received was burnt and tasted terrible."
print("=" * 60)
print(build_few_shot_prompt(complaint2))


# Add a new labelled example
add_example(
    "The restaurant sent noodles instead of the pasta I ordered.",
    "Wrong Item"
)

# Confirm the new example appears
complaint3 = "My order contained the wrong dish."
print("=" * 60)
print("Prompt after adding a new example:")
print(build_few_shot_prompt(complaint3))

Classify the food delivery complaint into one of these categories:
Late Delivery, Wrong Item, Missing Item, Poor Quality

Input: My order arrived 45 minutes late.
Output: Late Delivery

Input: I ordered a veg burger but received a chicken burger.
Output: Wrong Item

Input: My order was missing the fries I paid for.
Output: Missing Item

Input: The food was cold and tasted stale.
Output: Poor Quality

Input: My delivery arrived one hour after the promised time.
Output:
Classify the food delivery complaint into one of these categories:
Late Delivery, Wrong Item, Missing Item, Poor Quality

Input: My order arrived 45 minutes late.
Output: Late Delivery

Input: I ordered a veg burger but received a chicken burger.
Output: Wrong Item

Input: My order was missing the fries I paid for.
Output: Missing Item

Input: The food was cold and tasted stale.
Output: Poor Quality

Input: The pizza I received was burnt and tasted terrible.
Output:
Prompt after adding a new example:
Classify the food del

In [3]:
# Task 3: Semantic Search Over Restaurant FAQs

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Define restaurant FAQ entries
faqs = [
    "How long does delivery usually take?",
    "Can I cancel my order after placing it?",
    "What is the refund policy and when am I eligible for a refund?",
    "Can I request a substitution if an item is unavailable?",
    "How can I contact customer support for help?",
    "What should I do if my order is missing an item?",
    "Can I change my delivery address after placing the order?",
    "What happens if my food arrives cold or damaged?"
]

# 2. Load the Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Generate embeddings for all FAQs
faq_embeddings = model.encode(faqs)

# Convert embeddings to float32 NumPy array
faq_embeddings = np.asarray(faq_embeddings, dtype=np.float32)

# 4. Build FAISS IndexFlatL2
dimension = faq_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(faq_embeddings)

# Confirm index size
print("Number of FAQs:", len(faqs))
print("FAISS index size:", index.ntotal)

assert index.ntotal == len(faqs)


# 5. Semantic search function
def search_faq(query, k=2):
    query_embedding = model.encode([query])
    query_embedding = np.asarray(query_embedding, dtype=np.float32)

    distances, indices = index.search(query_embedding, k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append((faqs[idx], float(distance)))

    return results


# 6. Test with different wording
queries = [
    "How do I get my money back?",
    "My meal hasn't arrived yet, when should I expect it?"
]

for query in queries:
    print("\n" + "=" * 60)
    print("Query:", query)

    results = search_faq(query, k=2)

    for faq, distance in results:
        print(f"Distance: {distance:.4f}")
        print("FAQ:", faq)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of FAQs: 8
FAISS index size: 8

Query: How do I get my money back?
Distance: 1.0139
FAQ: What is the refund policy and when am I eligible for a refund?
Distance: 1.0535
FAQ: How can I contact customer support for help?

Query: My meal hasn't arrived yet, when should I expect it?
Distance: 0.9643
FAQ: How long does delivery usually take?
Distance: 1.0074
FAQ: What happens if my food arrives cold or damaged?


In [4]:
# Task 4: RAG-Powered Food Delivery Policy Q&A Pipeline

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import re


# ---------------------------------------------------------
# 1. Multi-paragraph food delivery policy (300+ words)
# ---------------------------------------------------------

policy_text = """
Refund Policy

Customers may request a refund when an order contains missing items,
incorrect items, or food that is substantially different from what was
ordered. For a missing item, customers should report the issue through
the support system within 24 hours of delivery. The support team may
issue a full or partial refund depending on the value of the missing
item. If the entire order is missing, the customer may be eligible for
a full refund.

For incorrect items, customers should provide the order number and
describe the item received. When the restaurant sends a different item
from the one ordered, the customer may receive a refund for the affected
item. Refund requests may be reviewed using the order details and
available delivery information. Refunds are normally returned to the
original payment method. Processing time may vary depending on the
payment provider.

Delivery Windows

Standard deliveries are normally expected to arrive within 30 to 45
minutes after the restaurant accepts the order. During periods of high
demand, severe weather, traffic congestion, or other unexpected events,
delivery may take longer. The estimated delivery time displayed to the
customer is an estimate and may change while the order is being prepared
and delivered.

If an order is significantly delayed beyond its estimated delivery
window, customers should contact support. Support may investigate the
delivery status and determine whether compensation or another resolution
is appropriate. Customers should provide their order number when
contacting support about a delayed delivery.

Order Cancellation

Customers may cancel an order before the restaurant begins preparing
the food. If cancellation occurs before preparation starts, the customer
will normally receive a full refund. Once the restaurant has started
preparing the order, cancellation may not be eligible for a refund
because ingredients and preparation resources may already have been
committed.

If the restaurant has accepted the order but preparation has not begun,
customers should submit the cancellation request as soon as possible.
The system will check the current order status before confirming the
cancellation. Once food preparation has started, customers should
contact support if they believe there are exceptional circumstances.

Refund Processing

Approved refunds are generally returned to the original payment method.
The time required for a refund to appear depends on the payment provider
and can take several business days. Customers should keep their order
number and payment information available when contacting support about
a refund.

Support

For questions about refunds, delivery delays, or cancellations,
customers can contact customer support through the application's help
section. Support agents may request the order number, delivery details,
and a description of the issue before reviewing the case.
"""


# ---------------------------------------------------------
# 2. Split policy into overlapping chunks
# ---------------------------------------------------------

def chunk_text(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks


chunks = chunk_text(policy_text, chunk_size=100, overlap=20)

print("Total chunks:", len(chunks))


# ---------------------------------------------------------
# 3. Generate embeddings
# ---------------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)
embeddings = np.asarray(embeddings, dtype=np.float32)


# ---------------------------------------------------------
# 4. Create FAISS index
# ---------------------------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)


# ---------------------------------------------------------
# 5. Retrieval function
# ---------------------------------------------------------

def retrieve(query, k=3):
    query_embedding = model.encode([query])
    query_embedding = np.asarray(query_embedding, dtype=np.float32)

    distances, indices = index.search(query_embedding, k)

    return [chunks[i] for i in indices[0]]


# ---------------------------------------------------------
# 6. Build structured RAG prompt
# ---------------------------------------------------------

def build_rag_prompt(query, retrieved_chunks):
    prompt = """
SYSTEM ROLE:
You are a food delivery policy support assistant.
Answer customer questions accurately and clearly using only the
provided policy context.

RETRIEVED CONTEXT:
"""

    for i, chunk in enumerate(retrieved_chunks, start=1):
        prompt += f"\nContext {i}:\n{chunk}\n"

    prompt += f"""
USER QUESTION:
{query}

INSTRUCTION:
Answer the question only from the provided context.
Do not use outside knowledge.
If the answer is not present in the provided context, say:
"I don't know."
"""

    return prompt


# ---------------------------------------------------------
# 7. End-to-end demonstration
# ---------------------------------------------------------

query = "What is the refund policy for missing items?"

retrieved_chunks = retrieve(query, k=3)

print("\n" + "=" * 70)
print("RETRIEVED CHUNKS")
print("=" * 70)

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\nContext {i}:\n{chunk}")


rag_prompt = build_rag_prompt(query, retrieved_chunks)

print("\n" + "=" * 70)
print("ASSEMBLED RAG PROMPT")
print("=" * 70)
print(rag_prompt)


# Placeholder for an actual LLM response
simulated_answer = (
    "Customers should report missing items within 24 hours of delivery. "
    "The support team may issue a full or partial refund depending on "
    "the value of the missing item."
)

print("\n" + "=" * 70)
print("SIMULATED ANSWER")
print("=" * 70)
print(simulated_answer)

Total chunks: 6


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index size: 6

RETRIEVED CHUNKS

Context 1:
Refund Policy Customers may request a refund when an order contains missing items, incorrect items, or food that is substantially different from what was ordered. For a missing item, customers should report the issue through the support system within 24 hours of delivery. The support team may issue a full or partial refund depending on the value of the missing item. If the entire order is missing, the customer may be eligible for a full refund. For incorrect items, customers should provide the order number and describe the item received. When the restaurant sends a different item from the one

Context 2:
The system will check the current order status before confirming the cancellation. Once food preparation has started, customers should contact support if they believe there are exceptional circumstances. Refund Processing Approved refunds are generally returned to the original payment method. The time required for a refund to appear dep